# Direct image → music

No ML or training. Original image → resized 16 × 16 luminance → note amplitudes. Top = higher pitch; left to right = time. This discards color and fine detail. Uses the same scales, sine synthesis, 0.15-second time grid, and peak normalization as the scripts.

Run top to bottom. Then change the playback controls to compare more paintings.


**Playback update:** argmax selects one pitch per column; matching consecutive pitches sustain. No chords, sampling, or melody smoothing. The model and training are unchanged.


In [1]:
%pip install numpy pillow matplotlib ipython

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# Works from the repository root or musician/.
HERE = next(p / "musician" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "musician/experiment_utils.py").is_file())
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
from experiment_utils import DEFAULT_DATA, image_paths, load_image, export_sound

DATA = DEFAULT_DATA  # Or DEFAULT_DATA / "oil-paint".
paths = image_paths(DATA)
print(f"Found {len(paths)} images in {DATA}")


Found 117 images in /Users/sarinali/Projects/3d-painting/data/monet/by-medium


In [3]:
EXPERIMENT = "direct-image"

def get_result(index):
    pixels = load_image(paths[index], size=16)
    luminance = pixels @ np.array([0.2126, 0.7152, 0.0722])
    return np.flipud(luminance).copy(), None


## Browse and listen

Change `START` to 6 for the next six paintings. Re-run the controls and display cells. The preview uses a 64 × 64 RGB image; the sound uses 16 × 16 luminance. WAV files are saved in `output/direct-image/`.

In [4]:
# Rerun this cell and the display cell to browse or change sound. No training needed.
START = 0
COUNT = 6
SCALE = "major_pentatonic"  # major, minor, minor_pentatonic, chromatic
ROOT_MIDI = 48  # C3
THRESHOLD = 0.0  # Increase to silence faint notes.


In [ ]:
from pathlib import Path
import sys, importlib
HERE = next(p / "musician" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "musician/experiment_utils.py").is_file())
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import experiment_utils
importlib.reload(experiment_utils)  # Pick up playback updates in an existing kernel.
from experiment_utils import sonify, melody_events, synthesize_events, export_sound

assert START >= 0 and COUNT > 0
indices = list(range(START, min(START + COUNT, len(paths))))
assert indices, "START is past the last image."
OUTPUT = HERE / "output" / EXPERIMENT
OUTPUT.mkdir(parents=True, exist_ok=True)
for index in indices:
    grid, reconstruction = get_result(index)
    columns = 3 if reconstruction is not None else 2
    fig, axes = plt.subplots(1, columns, figsize=(4 * columns, 3.5))
    axes[0].imshow(load_image(paths[index]))
    axes[0].set_title(f"#{index} · {paths[index].parent.name}")
    axes[0].axis("off")
    if reconstruction is not None:
        axes[1].imshow(reconstruction)
        axes[1].set_title("Reconstruction")
        axes[1].axis("off")
    axes[-1].imshow(grid, origin="lower", aspect="auto", vmin=0, vmax=1)
    events = melody_events(grid, SCALE, ROOT_MIDI, THRESHOLD)
    for event in events:
        if event['row'] is not None:
            axes[-1].hlines(event['row'], event['start'] - 0.5,
                            event['start'] + event['steps'] - 0.5, color="white", linewidth=2)
    axes[-1].set(title=f"Melody · {SCALE}", xlabel="Time step", ylabel="Pitch row")
    plt.tight_layout()
    plt.show()
    # Include controls in filenames to keep different listening experiments separate.
    name = f"melody-{index:03d}-{SCALE}-root{ROOT_MIDI}-threshold{THRESHOLD:g}.wav"
    wav = OUTPUT / name
    export_sound(grid, wav, SCALE, ROOT_MIDI, THRESHOLD)
    print(paths[index].name)
    display(Audio(filename=str(wav)))
